<a href="https://colab.research.google.com/github/emmanuelle-bul/AI_Exercises/blob/main/DocumentAnalysis_Summarize%26KeyWords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
from pathlib import Path
from pydantic import BaseModel

# functions

# от целия response извлича само отговора форматиран
def extract_summary(message):
  result = []
  for item in message.output:
    if item.type == 'message':
      for block in item.content:
        if block.type == 'output_text':
          result.append(block.text)
    return "\n\n\n".join(result)

# подобно обхождане на response, но написано в стил Python с използвани list comprehensions, за да извлече само ключовите думи
def extract_keywords(response):
  result = [block.text for item in response.output if item.type == 'message' for block in item.content if block.type == 'output_text']
  return "\n\n\n".join(result)



openai_api_key = userdata.get('OPENAI_API_KEY') #взимаме по име Secret (API key), с който ще достъпваме сървъра на OpenAI
openai_client = OpenAI(api_key=openai_api_key) #създаваме клиент

# със същото SDK на OpenAI можем да ползваме клиент към OpenRouter
openrouter_api_key = userdata.get('OPENROUTER_API_KEY') #взимаме по име Secret (API key), с който ще достъпваме сървъра на OpenRouter
openrouter_client = OpenAI(api_key=openrouter_api_key, base_url='https://openrouter.ai/api/v1') #създаваме клиент


In [ ]:
openai_client.models.list().data

[Model(id='gpt-4-turbo', created=1712361441, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-audio-2025-08-28', created=1756256146, object='model', owned_by='system', shutdown_date=None),
 Model(id='o4-mini', created=1744225351, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-3.5-turbo-0125', created=1706048358, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-4o-mini-2024-07-18', created=1721172717, object='model', owned_by='system', shutdown_date=None),
 Model(id='gpt-live-transcribe', created=1785168034, object='model', owned_by='system', shutdown_date=None),
 Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal', shutdown_date='2027-02-26'),
 Model(id='o1-pro', created=1742251791, object='model', owned_by='system', shutdown_date='2026-10-23'),
 Model(id='gpt-5-nano', created=1754426384, object='model', owned_by='system', shutdown_date=None),
 Model(id='tts-1-11

# Upload в сървъра на OpenAI

In [ ]:
path_to_file = Path("/content/BetterMe.pdf")

# upload в сървъра на OpenAI - https://platform.openai.com/storage/files
# можем да гп изтрием с Delete, за да не трупаме излишни файлове

file = openai_client.files.create(
  file=path_to_file.open("rb"),
  purpose="user_data"
  )

FileNotFoundError: [Errno 2] No such file or directory: '/content/BetterMe.pdf'

In [ ]:
file.id   # Когато разглеждаме response на части или отделните му параметри -> не изхабява тоукъни, само при извикването на responses(files).create

'file-Pi7VFYHUprv2iLNMbkRvmt'

# Summarize text

In [ ]:
summarize_response = openai_client.responses.create(
    model="gpt-5.4-nano",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
                {
                    "type": "input_text",
                    "text": "Обобщи цялата информация във файла",
                },
            ],
        }
    ],
)


In [ ]:
summary = extract_summary(summarize_response)



In [ ]:
summary

'Ето обобщение на цялата информация във файла (PDF “The Future of Personalized Health Coaching” за BetterMe):\n\n- **Кому е насочено:** PDF-ът приветства потребителя, който е започнал здравното си пътуване с **BetterMe**, и обяснява как приложението работи като **персонализирано решение**, подкрепящо ролята на **здравен коуч, личен треньор и мотивационен говорител**—всичко съобразено с целите и начина на живот на човека.\n\n- **Основна идея:** Трудността обикновено идва от **липса на план** и от множеството съвети “за всеки случай”. BetterMe “държи за ръка”, насочва и помага да не се изпада в несигурност или отказ.\n\n## 1) Проследяване на прогрес (Track your progress)\n- Без добро проследяване лесно се губи фокус и се правят грешки (напр. “скрито” увеличаване на калориите).\n- **Какво може да се следи в BetterMe:**\n  - **Калории** (с голяма база и barcode scanner)\n  - **Тегло/мерки** (без прекомерна фиксация само в кантара)\n  - **Стъпки**\n  - **Вода**\n  - **Работoти/тренировки**\

# File Delete от сървърите на OpenAI

In [ ]:
# openai_client.files.delete(file_id = file.id)

FileDeleted(id='file-Pi7VFYHUprv2iLNMbkRvmt', deleted=True, object='file')

# Extract keywords

In [ ]:
extract_keywords_response = openai_client.responses.create(
    model="gpt-5-mini",
    input=[
        { "role": "system", "content": "Ти си експерт, който извлича ключови думи от текст. При подадена обобщена информация за текст, извлечи около 10 думи, които са най-важни от текста, като се фокусираш върху тематиката."},
        { "role": "user", "content": f"Извлечи ключовите думи от тази обобщена информация: \n\n {summary}"}
    ],
    reasoning={"effort":"high"}   # означава -> Мисли по-задълбочено по тази задача!!! т.е. определяме колко "усилия" да вложи в нея. Даваме по-голяма свобода на модела. Очакваме повече време и токъни да са изразходвани.
)

In [ ]:
keywords = extract_keywords(extract_keywords_response)  # function that formats response

In [ ]:
print(keywords) # В какъв вид ще излезе резултата (форматиран или не) нямаме влияние - както реши моделът

extract_keywords_response.usage

Ето ключовите думи (≈10):

- BetterMe  
- персонализация  
- здравен коуч  
- проследяване  
- носима гривна (BetterMe Band)  
- тренировки  
- фастинг  
- хранене  
- ментално здраве  
- предизвикателства


ResponseUsage(input_tokens=1739, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=1763, output_tokens_details=OutputTokensDetails(reasoning_tokens=1664), total_tokens=3502)

## Categorize (Structured output)

In [ ]:
class Document_Categorization(BaseModel):
  category: str
  confidence: float

In [ ]:
CATEGORYZE_SYSTEM_PROMPT = """
Ти си експертен AI агент за анализ и категоризиране на документи, свързани с хранене, хранителни режими, диети, нутриция, здравословно хранене и планиране на хранене.

ТВОЯТА ЗАДАЧА:
Ще получиш обобщена информация от документ-източник. На база единствено на предоставеното обобщение трябва да определиш доколко информацията в документа е приложима и използваема за хранене и хранителни режими.

ОЦЕНЯВАНЕ:
Определи процент на използваемост на документа от 0 до 100%.

Процентът трябва да отразява:
- доколко документът съдържа информация, пряко свързана с хранене и нутриция;
- доколко информацията може да бъде използвана при създаване или персонализиране на хранителни режими;
- доколко информацията е конкретна и практическа;
- доколко информацията съдържа хранителни стойности, продукти, количества, рецепти, хранителни ограничения, режими или други релевантни данни;
- доколко документът може да подпомогне вземането на решения, свързани с хранене.

СКАЛА:
0–10% → Практически нерелевантен документ.
11–30% → Слабо релевантен; съдържа малко полезна информация.
31–50% → Частично релевантен; има значимо количество информация, но голяма част не е приложима.
51–70% → Релевантен; съдържа полезна информация за хранене и режими.
71–90% → Силно релевантен; голяма част от документа може да бъде използвана.
91–100% → Изключително релевантен; документът е основно или изцяло предназначен за хранене/хранителни режими.

ВАЖНИ ПРАВИЛА:
1. Не оценявай качеството или медицинската достоверност на информацията.
2. Оценявай единствено използваемостта на информацията спрямо хранене и хранителни режими.
3. Не измисляй информация, която не присъства в предоставеното обобщение.
4. Ако информацията е недостатъчна, намали увереността си.
5. Разграничавай пряко релевантна от косвено релевантна информация.
6. Общи теми като спорт, здраве или начин на живот не трябва автоматично да се считат за информация за хранене.
7. Ако документът съдържа само малка секция за хранене, процентът трябва да отразява реалната използваемост на цялата информация.

ВХОД:
Ще получиш:
SUMMARY: обобщение на документа-източник.

ИЗХОД:
Върни резултата САМО като JSON със следната структура:

{
    "usability_percentage": 0,
    "category": "",
    "confidence": 0,
    "reason": "",
    "relevant_information": []
}

ПОЛЕТА:
- usability_percentage: цяло число от 0 до 100.
- category: една от:
  "Нерелевантен",
  "Слабо релевантен",
  "Частично релевантен",
  "Релевантен",
  "Силно релевантен",
  "Изключително релевантен"
- confidence: увереност в оценката от 0 до 100.
- reason: кратко обяснение на причината за оценката.
- relevant_information: списък с основните типове информация от документа, които могат да бъдат използвани за хранене или хранителни режими.

Не добавяй текст извън JSON.
"""

categorize_response = openrouter_client.responses.parse(
    model= "deepseek/deepseek-v4-flash",
    input=[
        { "role": "system", "content": CATEGORYZE_SYSTEM_PROMPT},
        { "role": "user", "content":f"Определи категорията на документа базиран на неговата обобщена информация:\n\n{summary} "}
    ],
    text_format= Document_Categorization
)

In [ ]:
categorize_response.output_parsed


Document_Categorization(category='Релевантен', confidence=70.0)